# ROI Mapping Tutorial: Per-Modality Mapping with JSON Configs

This notebook demonstrates how to use PyTheranostics' per-modality ROI mapping features.

## Why Per-Modality Mapping?

In theranostics workflows, you often have:
- **CT masks** for morphology/volume (labeled with `_m` suffix)
- **SPECT masks** for activity quantification (labeled with `_a` suffix)

Both modalities may have ROIs that map to the same canonical target (e.g., `Kidney_Left`), but using different source names:
- CT: `Kidney_L_m` → `Kidney_Left`
- SPECT: `Kidney_L_a` → `Kidney_Left`

If you apply the same mapping to both studies, you can create conflicts where multiple sources map to the same target within a single study.

**Solution:** PyTheranostics provides modality-aware mapping tools that:
1. Filter mappings to only keys present in each study
2. Detect conflicts automatically
3. Support JSON config files for reusability

## Setup: Import Libraries

In [ ]:
import pytheranostics as tx
from pytheranostics.imaging_ds.longitudinal_study import LongitudinalStudy

# For this tutorial, we'll assume you have CT and SPECT longitudinal studies loaded
# Example (replace with your actual data paths):
# longCT, longSPECT, inj, used_mappings = tx.imaging_ds.create_studies_with_masks(
#     storage_root="/path/to/data",
#     patient_id="PATIENT_ID",
#     cycle_no=1,
#     parallel=True,
# )

## Approach 1: Inline Dictionaries (Quick Start)

Define mappings directly in your notebook code.

In [ ]:
# Define modality-specific mappings
ct_mask_mapping = {
    "Liver": "Liver",
    "Skeleton": "Skeleton",
    "Kidney_L_m": "Kidney_Left",
    "Kidney_R_m": "Kidney_Right",
    "Spleen": "Spleen",
    "Bladder": "Bladder",
    "Parotid_L_m": "ParotidGland_Left",
    "Parotid_R_m": "ParotidGland_Right",
    "Submandibular_L_m": "SubmandibularGland_Left",
    "Submandibular_R_m": "SubmandibularGland_Right",
    "WBCT": "WholeBody",
}

spect_mask_mapping = {
    "Liver": "Liver",
    "Skeleton": "Skeleton",
    "Kidney_L_a": "Kidney_Left",
    "Kidney_R_a": "Kidney_Right",
    "Spleen": "Spleen",
    "Bladder": "Bladder",
    "Parotid_L_a": "ParotidGland_Left",
    "Parotid_R_a": "ParotidGland_Right",
    "Submandibular_L_a": "SubmandibularGland_Left",
    "Submandibular_R_a": "SubmandibularGland_Right",
    "WBCT": "WholeBody",
}

# Apply mappings (replace longCT and longSPECT with your actual study variables)
# result = LongitudinalStudy.apply_per_modality_mappings(
#     ct_study=longCT,
#     spect_study=longSPECT,
#     ct_mask_mapping=ct_mask_mapping,
#     spect_mask_mapping=spect_mask_mapping,
# )

# # Check results
# print(f"✓ CT mappings applied: {len(result['ct_applied'])}")
# print(f"✓ SPECT mappings applied: {len(result['spect_applied'])}")
# if result['ct_conflicts'] or result['spect_conflicts']:
#     print("⚠️ Conflicts detected!")
#     print(f"  CT conflicts: {result['ct_conflicts']}")
#     print(f"  SPECT conflicts: {result['spect_conflicts']}")

## Approach 2: JSON Config File (Recommended for Projects)

### Step 1: Create a JSON Config File

Create a file named `roi_mappings.json` in your project directory with this structure:

```json
{
  "_comment": "ROI mapping configuration for your project",
  
  "ct_mappings": {
    "Liver": "Liver",
    "Skeleton": "Skeleton",
    "Kidney_L_m": "Kidney_Left",
    "Kidney_R_m": "Kidney_Right",
    "Spleen": "Spleen",
    "Bladder": "Bladder",
    "Parotid_L_m": "ParotidGland_Left",
    "Parotid_R_m": "ParotidGland_Right",
    "Submandibular_L_m": "SubmandibularGland_Left",
    "Submandibular_R_m": "SubmandibularGland_Right",
    "WBCT": "WholeBody"
  },
  
  "spect_mappings": {
    "Liver": "Liver",
    "Skeleton": "Skeleton",
    "Kidney_L_a": "Kidney_Left",
    "Kidney_R_a": "Kidney_Right",
    "Spleen": "Spleen",
    "Bladder": "Bladder",
    "Parotid_L_a": "ParotidGland_Left",
    "Parotid_R_a": "ParotidGland_Right",
    "Submandibular_L_a": "SubmandibularGland_Left",
    "Submandibular_R_a": "SubmandibularGland_Right",
    "WBCT": "WholeBody"
  }
}
```

**Tip:** A template is available at `pytheranostics/data/roi_mappings_template.json` — copy and customize it!

### Step 2a: Load JSON After Studies Created

In [ ]:
# Load mappings from JSON config file
json_config_path = "roi_mappings.json"  # <-- Path to your JSON file

# Uncomment to use:
# mappings = LongitudinalStudy.load_mappings_from_json(json_config_path)
# 
# result = LongitudinalStudy.apply_per_modality_mappings(
#     ct_study=longCT,
#     spect_study=longSPECT,
#     ct_mask_mapping=mappings['ct_mappings'],
#     spect_mask_mapping=mappings['spect_mappings'],
# )
# 
# print(f"✓ Applied {len(result['ct_applied'])} CT mappings")
# print(f"✓ Applied {len(result['spect_applied'])} SPECT mappings")
# 
# # Show what was mapped
# print("\nCT mappings:")
# for src, dst in sorted(result['ct_applied'].items()):
#     if src != dst:
#         print(f"  {src} → {dst}")
# 
# print("\nSPECT mappings:")
# for src, dst in sorted(result['spect_applied'].items()):
#     if src != dst:
#         print(f"  {src} → {dst}")

### Step 2b: Pass JSON Config During Data Loading (One-Step Approach)

You can apply mappings automatically when loading data by passing `mapping_config` to `create_studies_with_masks()`.

In [ ]:
# Load data AND apply mappings in one step
# Uncomment and customize:

# longCT, longSPECT, inj, used_mappings = tx.imaging_ds.create_studies_with_masks(
#     storage_root="/path/to/data",
#     patient_id="PATIENT_ID",
#     cycle_no=1,
#     parallel=True,
#     mapping_config="roi_mappings.json",  # <-- Mappings applied automatically!
# )
# 
# # Unpack injection metadata
# InjectionDate = inj.get("InjectionDate") or ""
# InjectionTime = inj.get("InjectionTime") or ""
# InjectedActivity = str(inj.get("InjectedActivity")) if inj.get("InjectedActivity") is not None else ""
# PatientWeight_g = inj.get("PatientWeight_g") if inj.get("PatientWeight_g") is not None else None
# 
# print(f"✓ CT timepoints: {len(longCT.images)}")
# print(f"✓ SPECT timepoints: {len(longSPECT.images)}")
# print(f"✓ Mappings applied at load time from JSON config")

## Understanding the Result Dictionary

The `apply_per_modality_mappings()` function returns a detailed diagnostic dictionary:

In [ ]:
# Example result structure (uncomment to see with your data):
# result = {
#     'ct_applied': {          # Mappings actually applied to CT
#         'Kidney_L_m': 'Kidney_Left',
#         'Liver': 'Liver',
#         ...
#     },
#     'spect_applied': {       # Mappings actually applied to SPECT
#         'Kidney_L_a': 'Kidney_Left',
#         'Liver': 'Liver',
#         ...
#     },
#     'ct_absent': [],         # CT mapping keys not found in CT masks
#     'spect_absent': [],      # SPECT mapping keys not found in SPECT masks
#     'ct_conflicts': {},      # CT conflicts: {target: [source1, source2]}
#     'spect_conflicts': {},   # SPECT conflicts: {target: [source1, source2]}
# }

# Check for issues:
# if result['ct_absent']:
#     print(f"⚠️ CT mappings provided but not found: {result['ct_absent']}")
# 
# if result['spect_absent']:
#     print(f"⚠️ SPECT mappings provided but not found: {result['spect_absent']}")
# 
# if result['ct_conflicts']:
#     print(f"❌ CT conflicts detected: {result['ct_conflicts']}")
#     print("   Fix: remove duplicate mappings to the same target")
# 
# if result['spect_conflicts']:
#     print(f"❌ SPECT conflicts detected: {result['spect_conflicts']}")
#     print("   Fix: remove duplicate mappings to the same target")

## Adding Lesion Mappings

You can add lesion or other one-off mappings using `manual_overrides`:

## Configuration: Valid Organ Names

### Overview

The `voi_mappings_config.json` file now includes a `valid_organ_names` section that defines which organ/VOI names are accepted as valid targets for mappings. This prevents invalid mappings from silently failing.

### Why This Matters

When you create a mapping like:
```json
"spect_mappings": {
  "Kidney_L_a": "Kidney_L"
}
```

The system validates that `"Kidney_L"` is in the list of `valid_organ_names`. If it's not, the mapping is rejected and the ROI stays unmapped (identity mapping). This helps catch typos and naming inconsistencies early.

### Default Valid Organ Names (OLINDA-Compatible)

The package template includes these OLINDA-compatible organ names by default. You can customize this list for your project:

```json
{
  "valid_organ_names": {
    "_description": "List of valid organ/VOI names for validation. These default names are compatible with OLINDA dosimetry calculations. Users can add custom organs as needed for their workflows.",
    "names": [
      "Kidney_Left",
      "Kidney_Right",
      "Liver",
      "Spleen",
      "Bladder",
      "SubmandibularGland_Left",
      "SubmandibularGland_Right",
      "ParotidGland_Left",
      "ParotidGland_Right",
      "BoneMarrow",
      "Skeleton",
      "WholeBody",
      "RemainderOfBody",
      "TotalTumorBurden"
    ]
  }
}
```

### Customizing Valid Organ Names

If your project uses different naming conventions or custom organs, add them to your project's `voi_mappings_config.json`:

```json
{
  "valid_organ_names": {
    "_description": "Custom organ names for our project",
    "names": [
      "Kidney_Left",
      "Kidney_Right",
      "kidney_cyst_left",
      "kidney_cyst_right",
      "MyCustomOrgan",
      "Lesion_1",
      "Lesion_2"
    ]
  },
  "ct_mappings": {...},
  "spect_mappings": {...}
}
```

### Loading Valid Organ Names in Your Code

```python
from pytheranostics.imaging_ds import LongitudinalStudy

# Get the current valid organ names (loads from config or uses defaults)
valid_organs = LongitudinalStudy._get_valid_organ_names()
print("Valid organs:", valid_organs)
```

The system searches for `voi_mappings_config.json` in this order:
1. **Current directory** (your notebook location)
2. **One level up** (project root)
3. **Package template** (OLINDA defaults)

### Workflow Integration

When you call `create_studies_with_masks()` with a mapping config:

```python
longCT, longSPECT, inj, used = tx.imaging_ds.create_studies_with_masks(
    storage_root="./data",
    patient_id="PATIENT_ID",
    cycle_no=1,
    mapping_config="./voi_mappings_config.json"  # ← Loads valid_organ_names too
)
```

Both the mappings AND the valid organ names are loaded from your config file. This ensures consistency across your project.

## Configuration: Canonical Name Mappings

### Overview

The `canonical_mappings` section in `voi_mappings_config.json` defines automatic abbreviation normalization for the `auto_map=True` mode in `create_studies_with_masks()`. This is useful when your RTSTRUCT files use abbreviated names like `Kidney_L` but you want them automatically converted to the canonical form `Kidney_Left`.

### How It Works

When you set `auto_map=True`:

```python
longCT, longSPECT, inj, used = tx.imaging_ds.create_studies_with_masks(
    storage_root="./data",
    patient_id="PATIENT_ID",
    cycle_no=1,
    auto_map=True  # ← Enables automatic canonical name mapping
)
```

The system:
1. Loads canonical_mappings from your config file
2. For each ROI name in your RTSTRUCT:
   - Strips modality suffixes (`_m`, `_a`)
   - Looks up the base name in canonical_mappings
   - Maps to the canonical name if found
   - Keeps the name as-is if no mapping exists

### Default Canonical Mappings

The package template includes these common abbreviations:

```json
{
  "canonical_mappings": {
    "_description": "Best-effort ROI name normalization for auto_map mode. Maps abbreviated/common names to canonical organ names. Used when auto_map=True is set in create_studies_with_masks.",
    "mappings": {
      "Kidney_L": "Kidney_Left",
      "Kidney_R": "Kidney_Right",
      "Parotid_L": "ParotidGland_Left",
      "Parotid_R": "ParotidGland_Right",
      "Submandibular_L": "SubmandibularGland_Left",
      "Submandibular_R": "SubmandibularGland_Right",
      "WBCT": "WholeBody"
    }
  }
}
```

### Customizing Canonical Mappings

Add custom mappings to your project's `voi_mappings_config.json` for your institution's naming conventions:

```json
{
  "canonical_mappings": {
    "_description": "Custom mappings for our institution",
    "mappings": {
      "Kidney_L": "Kidney_Left",
      "Kidney_R": "Kidney_Right",
      "KL": "Kidney_Left",
      "KR": "Kidney_Right",
      "Parotid_L": "ParotidGland_Left",
      "Parotid_R": "ParotidGland_Right",
      "Liver_N": "Liver",
      "Liver_C": "Liver",
      "WBCT": "WholeBody"
    }
  }
}
```

### When to Use: auto_map vs Explicit Mapping

| Scenario | Approach | Example |
|----------|----------|---------|
| **Known abbreviations, consistent naming** | `auto_map=True` | RTSTRUCT names are always `Kidney_L`, `Kidney_R`, etc. → auto converts to canonical |
| **Modality-specific names** | Explicit `ct_mappings`/`spect_mappings` | CT has `Kidney_L_m`, SPECT has `Kidney_L_a` → different mappings per modality |
| **Complex/variable naming** | Explicit mappings in config | ROIs named inconsistently across projects → use full `ct_mappings`/`spect_mappings` |
| **Mixed approach** | `auto_map=True` + explicit overrides | Use auto_map for most, but override specific conflicting names with explicit mappings |

### Example Workflow: Auto-Map with Suffix Stripping

```python
# RTSTRUCT contains: Kidney_L_a, Kidney_R_a, Kidney_L_m, Kidney_R_m

longCT, longSPECT, inj, used = tx.imaging_ds.create_studies_with_masks(
    storage_root="./data",
    patient_id="PATIENT_ID",
    cycle_no=1,
    auto_map=True  # Enables canonical_mappings
)

# Results (from used_mappings):
# CT:    Kidney_L_m → Kidney_Left,  Kidney_R_m → Kidney_Right
# SPECT: Kidney_L_a → Kidney_Left,  Kidney_R_a → Kidney_Right
```

The system:
1. Strips `_m` and `_a` suffixes → `Kidney_L`, `Kidney_R`
2. Looks up in canonical_mappings → finds `Kidney_Left`, `Kidney_Right`
3. Validates against valid_organ_names → passes
4. Applies the mapping ✓

In [ ]:
# Load base mappings from JSON
# mappings = LongitudinalStudy.load_mappings_from_json("roi_mappings.json")

# Add lesion mappings (apply to both CT and SPECT)
# lesion_overrides = {
#     "Lesion1_CT": "Lesion_1",
#     "Lesion1_SPECT": "Lesion_1",
#     "Lesion2": "Lesion_2",
# }

# result = LongitudinalStudy.apply_per_modality_mappings(
#     ct_study=longCT,
#     spect_study=longSPECT,
#     ct_mask_mapping=mappings['ct_mappings'],
#     spect_mask_mapping=mappings['spect_mappings'],
#     manual_overrides=lesion_overrides,  # <-- Added here
# )

## Best Practices

1. **Use JSON configs for projects** — easier to version control and share with collaborators
2. **Start from the template** — copy `pytheranostics/data/roi_mappings_template.json` and customize
3. **Check diagnostics** — always inspect `result['ct_conflicts']` and `result['spect_conflicts']`
4. **Review absent keys** — `result['ct_absent']` and `result['spect_absent']` may indicate typos
5. **Use consistent suffixes:**
   - `_m` for CT/morphology ROIs (mass/volume)
   - `_a` for SPECT/activity ROIs
6. **Version control your JSON** — commit `roi_mappings.json` to your repository

## Valid Canonical Target Names

PyTheranostics recognizes these canonical organ names:
- `Kidney_Left`, `Kidney_Right`
- `Liver`
- `Spleen`
- `Bladder`
- `ParotidGland_Left`, `ParotidGland_Right`
- `SubmandibularGland_Left`, `SubmandibularGland_Right`
- `BoneMarrow`
- `Skeleton`
- `WholeBody`
- `RemainderOfBody`
- `TotalTumorBurden`
- Lesions: `Lesion_1`, `Lesion_2`, etc. (format: `Lesion_N` where N is a positive integer)

## Summary: Three Ways to Apply Mappings

| Approach | When to Use | Code Example |
|----------|-------------|-------------|
| **Inline dicts** | Quick prototyping, one-off analysis | `result = LongitudinalStudy.apply_per_modality_mappings(ct_study=longCT, spect_study=longSPECT, ct_mask_mapping={...}, spect_mask_mapping={...})` |
| **JSON after load** | Reusable configs, already loaded data | `mappings = LongitudinalStudy.load_mappings_from_json("roi_mappings.json"); result = LongitudinalStudy.apply_per_modality_mappings(..., ct_mask_mapping=mappings['ct_mappings'], ...)` |
| **JSON during load** | One-step workflow, clean notebooks | `longCT, longSPECT, inj, used = tx.imaging_ds.create_studies_with_masks(..., mapping_config="roi_mappings.json")` |

## Next Steps

1. Copy the template: `pytheranostics/data/roi_mappings_template.json`
2. Customize it for your project's ROI naming conventions
3. Use one of the three approaches above in your analysis notebooks
4. Check the diagnostics to ensure mappings are correct
5. Proceed with dose calculations using the mapped ROI names

## Questions?

Check the inline documentation:
```python
help(LongitudinalStudy.apply_per_modality_mappings)
help(LongitudinalStudy.load_mappings_from_json)
```